## 任务：
撰写 Python 代码，统计 https://www.lianxh.cn 的推文阅读量。呈现如下结果：
- 网站总阅读量 ('"pv":1037' 表示阅读量为 1037)
- 不同类别的推文的总阅读量

以下分析不包含作者为 '连享会' 的数据

- 总阅读量最高的前 30 位作者，以及他们的总阅读量和平均阅读量。
  - 列表呈现格式为：Author, Total Hits, number-of-blogs, Average Hits
- 每个年度阅读量最高的 10 篇推文的标题、作者、阅读量和发布年份。输出格式为：
   '''
   ### 各年度阅读量最高的 10 篇推文
   - 2026 
      - Author, Year, [blog-title](url).  Hits: ####
      - ……
   - 2025
     - Author, Year, [blog-title](url).  Hits: ####
   '''    
    
## api
- https://www.lianxh.cn/web-api/search?s=

## 输出

- 先使用 Markdown list 格式输出基本信息
- 再撰写一个通讯稿风格的 Markdown 总结

然后将上述结果写成一个可以发布的总结报告，Markdown 格式，输出 .md 文件，我会发布在 https://www.lianxh.cn 上。 


In [5]:
# 基于现有 df 和 author 变量生成可发布的 Markdown 报告
# 说明：
# 1) 网站总阅读量、分类阅读量：基于全部样本
# 2) 作者榜、年度 Top10：剔除作者为 author 的推文

exclude_author = author

# --- 全站统计（全样本） ---
total_read_all = int(df["阅读量"].sum())
total_articles_all = int(df.shape[0])
cat_read_all = (
    df.groupby("分类", dropna=False)["阅读量"]
    .sum()
    .sort_values(ascending=False)
)

# --- 剔除指定作者后的样本 ---
df_no_author = df.loc[df["作者"] != exclude_author].copy()

top_authors_30 = (
    df_no_author.groupby("作者")
    .agg(
        **{
            "Total Hits": ("阅读量", "sum"),
            "number-of-blogs": ("标题", "count"),
            "Average Hits": ("阅读量", "mean"),
        }
    )
    .sort_values(["Total Hits", "number-of-blogs", "Average Hits"], ascending=[False, False, False])
    .head(30)
    .reset_index()
    .rename(columns={"作者": "Author"})
)

years = sorted(df_no_author["发布年份"].dropna().astype(int).unique(), reverse=True)

yearly_top10 = {
    y: (
        df_no_author.loc[df_no_author["发布年份"] == y]
        .sort_values(["阅读量", "标题"], ascending=[False, True])
        .head(10)
        .copy()
    )
    for y in years
}

def md_escape(text):
    text = str(text)
    return text.replace("\\", "\\\\").replace("[", "\\[").replace("]", "\\]")

# --- 基本信息 Markdown list ---
basic_lines = [
    "# 连享会推文阅读量统计报告",
    "",
    "## 基本信息",
    f"- 网站总阅读量：**{total_read_all:,}**",
    f"- 推文总数：**{total_articles_all:,}**",
    f"- 分类数量：**{cat_read_all.shape[0]:,}**",
    "- 不同类别的推文总阅读量：",
]
basic_lines.extend([f"  - {k}: {int(v):,}" for k, v in cat_read_all.items()])
basic_lines.append(f"- 以下作者榜单与年度榜单分析已剔除作者 **{exclude_author}** 的推文。")

# --- 作者榜 ---
author_lines = [
    "",
    "## 总阅读量最高的前 30 位作者",
    "- 列表格式：Author, Total Hits, number-of-blogs, Average Hits",
]
author_lines.extend(
    [
        f"- {row['Author']}, {int(row['Total Hits']):,}, {int(row['number-of-blogs'])}, {row['Average Hits']:.2f}"
        for _, row in top_authors_30.iterrows()
    ]
)

# --- 各年度 Top 10 ---
year_lines = [
    "",
    "### 各年度阅读量最高的 10 篇推文",
]
for y, sub in yearly_top10.items():
    year_lines.append(f"- {y}")
    for _, row in sub.iterrows():
        year_lines.append(
            f"  - {row['作者']}, {int(row['发布年份'])}, "
            f"[{md_escape(row['标题'])}]({row['链接']}). Hits: {int(row['阅读量']):,}"
        )

# --- 通讯稿风格总结 ---
top_cat_name = str(cat_read_all.index[0])
top_cat_hits = int(cat_read_all.iloc[0])
top_author_name = str(top_authors_30.iloc[0]["Author"])
top_author_hits = int(top_authors_30.iloc[0]["Total Hits"])
top_author_avg = float(top_authors_30.iloc[0]["Average Hits"])
top_author_n = int(top_authors_30.iloc[0]["number-of-blogs"])
year_min = int(df_no_author["发布年份"].min())
year_max = int(df_no_author["发布年份"].max())

summary_lines = [
    "",
    "## 通讯稿摘要",
    "",
    f"统计结果显示，截至当前样本，连享会网站共收录推文 **{total_articles_all:,}** 篇，累计阅读量达到 **{total_read_all:,}**。"
    f" 从栏目表现看，**{top_cat_name}** 以 **{top_cat_hits:,}** 次阅读位居各类别前列，显示出该主题长期保持较高关注度。",
    "",
    f"在剔除作者 **{exclude_author}** 的推文后，作者影响力格局更加清晰。"
    f" 其中，**{top_author_name}** 以 **{top_author_hits:,}** 次总阅读量位列首位，累计发文 **{top_author_n}** 篇，"
    f" 平均每篇阅读量 **{top_author_avg:,.2f}**，表现出较强的持续输出能力与读者吸引力。",
    "",
    f"进一步按年度梳理 {year_min}—{year_max} 年各年度阅读量前 10 篇推文，可以较好刻画平台内容热点的演进路径。"
    f" 该榜单既可用于年度选题复盘，也可为课程内容组织、专题策划与教学案例整理提供参考。",
]

report_md = "\n".join(basic_lines + author_lines + year_lines + summary_lines)

output_file = "连享会推文阅读量统计报告.md"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md)
print(f"\nMarkdown 报告已保存为：{output_file}")

# 连享会推文阅读量统计报告

## 基本信息
- 网站总阅读量：**29,237,301**
- 推文总数：**1,746**
- 分类数量：**41**
- 不同类别的推文总阅读量：
  - 回归分析: 2,484,417
  - 数据处理: 2,482,920
  - 倍分法DID: 2,178,475
  - Stata绘图: 1,656,189
  - 面板数据: 1,504,259
  - 内生性-因果推断: 1,426,202
  - Stata命令: 1,358,059
  - 交乘项-调节-中介: 1,117,586
  - 工具软件: 1,052,689
  - 结果输出: 973,136
  - 论文写作: 895,831
  - 其它: 846,468
  - IV-GMM: 820,449
  - Python-R-Matlab: 798,928
  - Stata资源: 720,908
  - Stata教程: 684,980
  - Stata入门: 642,909
  - 专题课程: 582,904
  - 数据分享: 570,384
  - Markdown-LaTeX: 561,358
  - Probit-Logit: 533,152
  - Stata程序: 516,611
  - 计量专题: 501,179
  - PSM-Matching: 495,583
  - 断点回归RDD: 461,600
  - 时间序列: 382,240
  - 文本分析-爬虫: 371,881
  - SFA-DEA-效率分析: 335,342
  - 论文重现: 332,083
  - 空间计量-网络分析: 302,330
  - 机器学习: 302,241
  - 助教招聘: 249,201
  - AI专题: 230,710
  - 公开课: 203,657
  - 分位数回归: 173,700
  - 合成控制法: 153,936
  - 答疑-板书: 139,618
  - 往期课程: 73,980
  - 生存分析: 64,962
  - 未分类: 44,295
  - 风险管理: 9,949
- 以下作者榜单与年度榜单分析已剔除作者 **连享会** 的推文。

## 总阅读量最高的前 30 位作者
- 列表格式：Author, 